# Non-Functional Requirements for AI Agents

### A Practical Guide to Cost, Compute Footprint, and Environmental Impact

## Overview

When building software, requirements come in two flavours:

- **Functional requirements** - *what* the system does ("summarize a document", "answer a question").
- **Non-functional requirements (NFRs)** - *how well* it does it: speed, cost, reliability, and resource usage.

For traditional software, NFRs are well understood: measure CPU, memory, and response time; profile hot paths; right-size infrastructure. For AI agents, the picture is more complex.

### Why agents amplify NFR concerns

An AI agent is not a single LLM call - it is an **autonomous loop** that plans, calls tools, observes results, and iterates until a task is complete. Each loop iteration is a new LLM call. Each tool result is appended to the context window and re-sent on the next call. A task that seems like "one request" from the outside can easily generate 5–20 LLM calls under the hood.

This has direct consequences:

| NFR dimension | Single LLM call | Multi-step agent |
|---|---|---|
| **Token cost** | Predictable, bounded | Multiplied by steps; context grows each turn |
| **Latency** | One round-trip | N round-trips stacked in sequence |
| **Memory** | Stateless | Holds full conversation history in RAM |
| **Energy** | Per-query | Per-step, often 5–20× per task |

### What this notebook covers

1. **Cost** - token economics, measuring real spend, and optimisation strategies
2. **Compute footprint** - latency, throughput, and memory management
3. **Environmental footprint** - a brief, informed perspective on energy and carbon

Code examples use IBM Granite models via watsonx.ai.

**Prerequisites:** Python 3.11+, basic familiarity with LangChain/LangGraph.

**References:**
- [IBM Granite Models](https://www.ibm.com/granite)
- [Token Economics for LLM Agents (arXiv 2025)](https://arxiv.org/html/2605.09104v1)
- [FinOps for AI Overview](https://www.finops.org/wg/finops-for-ai-overview/)
- [Observability & Tracing notebook](../Tracing/Tracing_Agent.ipynb) - companion recipe for production monitoring

## Table of Contents

1. [Setup & Environment](#section-1)
2. [Cost: Understanding Token Economics](#section-2)
3. [Cost: Measuring Agent Spend in Practice](#section-3)
4. [Cost: Optimization Strategies](#section-4)
5. [Compute Footprint: Latency & Throughput](#section-5)
6. [Compute Footprint: Memory & Context Management](#section-6)
7. [Environmental Footprint](#section-7)
8. [Summary & Reusable NFR Profiler](#section-8)

<a id='section-1'></a>
## 1. Setup & Environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md).

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

This notebook is compatible with Python 3.12 and Python 3.11.

### 1.1 Install dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install \
    "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain_ibm \
    pandas \
    matplotlib \
    seaborn \
    rich \
    psutil \
    tiktoken
! echo "::endgroup::"

### 1.2 Import libraries

In [ ]:
import os
import time
import json
import psutil
from dataclasses import dataclass, field
from typing import Any, Optional

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from rich import print as rprint

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.utils.utils import convert_to_secret_str
from langchain_core.callbacks import UsageMetadataCallbackHandler

sns.set_theme(style="whitegrid", palette="muted")
print("Imports successful")

### 1.3 Initialize the Granite model

You will need `WATSONX_URL`, `WATSONX_APIKEY`, and `WATSONX_PROJECT_ID` from a [watsonx.ai instance](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb).

In [ ]:
from langchain.chat_models import init_chat_model
from ibm_granite_community.notebook_utils import get_env_var

MODEL_ID = "ibm/granite-4-h-small"
MODEL_PARAMETERS = {
    "temperature": 0,
    "max_completion_tokens": 300,
    "repetition_penalty": 1.05,
}

llm = init_chat_model(
    model=MODEL_ID,
    model_provider="ibm",
    url=convert_to_secret_str(get_env_var("WATSONX_URL")),
    apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
    project_id=get_env_var("WATSONX_PROJECT_ID"),
    params=MODEL_PARAMETERS,
)

rprint(f"[green]Connected to watsonx.ai — model: {MODEL_ID}[/green]")

<a id='section-2'></a>
## 2. Cost: Understanding Token Economics

### What is a token?

LLMs do not process text character-by-character or word-by-word, they process **tokens**, which are sub-word units produced by a tokenizer. As a rule of thumb:

> **1 token ≈ 0.75 English words** (roughly 4 characters)

So "The quick brown fox" is approximately 5 tokens, and a 500-word essay is roughly 670 tokens.

### How pricing works

LLM API providers charge separately for **input tokens** (what you send) and **output tokens** (what the model generates). Output tokens are typically more expensive because generating each token requires a full forward pass through the model.

The table below shows illustrative pricing tiers as of mid-2025. Actual prices vary by provider and change frequently.

| Model tier | Example | Input ($/1M tokens) | Output ($/1M tokens) |
|---|---|---|---|
| Small / efficient | Granite 4 Small, Haiku | ~$0.25 | ~$1.25 |
| Medium | Granite 4 Medium, Sonnet | ~$3 | ~$15 |
| Large / flagship | Granite 4 Large, Opus | ~$15 | ~$75 |

### The agent cost multiplier

In a multi-step agent, the **entire conversation history** is re-sent to the model on every turn. If your agent takes 5 steps and each step accumulates 200 tokens of context:

- Turn 1: 200 input tokens  
- Turn 2: 400 input tokens (history doubles)  
- Turn 3: 600 input tokens  
- Turn 4: 800 input tokens  
- Turn 5: 1000 input tokens  
- **Total input: 3000 tokens** — not 1000

This **quadratic growth** is why a 5-step agent can cost 3–5× more than a single call with the same total content.

In [ ]:
# TokenCostEstimator: reusable utility for cost calculations

@dataclass
class TokenCostEstimator:
    """Estimates LLM API cost from token counts."""
    input_price_per_m: float    # USD per 1M input tokens
    output_price_per_m: float   # USD per 1M output tokens
    model_label: str = "unknown"

    def estimate(self, input_tokens: int, output_tokens: int) -> dict:
        cost = (
            (input_tokens / 1_000_000) * self.input_price_per_m
            + (output_tokens / 1_000_000) * self.output_price_per_m
        )
        return {
            "model": self.model_label,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "estimated_cost_usd": round(cost, 6),
        }

# Illustrative pricing for three tiers
SMALL_MODEL  = TokenCostEstimator(0.25,  1.25,  "granite-small")
MEDIUM_MODEL = TokenCostEstimator(3.0,   15.0,  "granite-medium")
LARGE_MODEL  = TokenCostEstimator(15.0,  75.0,  "granite-large")

# Demonstrate cost across a typical range of token counts
scenarios = [
    ("Single chat turn",      150,  80),
    ("5-step agent task",    1200, 400),
    ("Complex research task", 5000, 800),
]

rows = []
for label, inp, out in scenarios:
    for estimator in [SMALL_MODEL, MEDIUM_MODEL, LARGE_MODEL]:
        result = estimator.estimate(inp, out)
        rows.append({"scenario": label, **result})

df_cost = pd.DataFrame(rows)
pivot = df_cost.pivot(index="scenario", columns="model", values="estimated_cost_usd")

# Reorder rows from simple to complex
scenario_order = ["Single chat turn", "5-step agent task", "Complex research task"]
pivot = pivot.reindex(scenario_order)

ax = pivot.plot(kind="bar", figsize=(10, 5), rot=0, colormap="viridis")
ax.set_title("Estimated cost by scenario and model tier", fontsize=14, pad=12)
ax.set_xlabel("Scenario")
ax.set_ylabel("Estimated cost (USD)")
ax.legend(title="Model tier")
for container in ax.containers:
    ax.bar_label(container, fmt="$%.5f", fontsize=7, padding=3)
plt.tight_layout()
plt.show()

rprint(pivot.to_string())

### Context growth: why agent costs compound

The cell below simulates how input tokens grow across turns as the conversation history accumulates. Each bar represents the number of input tokens sent to the model on that turn including all prior turns.

In [ ]:
# Simulate context accumulation over 6 agent turns
# Each turn adds ~150 tokens of new content (user query + tool result)

TOKENS_PER_TURN = 150   # new tokens added each turn
SYSTEM_PROMPT_TOKENS = 200  # fixed overhead
N_TURNS = 6

turns = list(range(1, N_TURNS + 1))
input_tokens_per_turn = [
    SYSTEM_PROMPT_TOKENS + TOKENS_PER_TURN * t for t in turns
]
cumulative_total = sum(input_tokens_per_turn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: tokens per turn
axes[0].bar(turns, input_tokens_per_turn, color=sns.color_palette("muted")[0])
axes[0].set_title("Input tokens sent per agent turn")
axes[0].set_xlabel("Agent turn")
axes[0].set_ylabel("Input tokens")
for i, v in enumerate(input_tokens_per_turn):
    axes[0].text(i + 1, v + 10, str(v), ha="center", fontsize=9)

# Right: cumulative cost comparison — naive (flat) vs actual (compounding)
cost_naive = [t * TOKENS_PER_TURN / 1_000_000 * SMALL_MODEL.input_price_per_m for t in turns]
cost_actual = [
    sum(input_tokens_per_turn[:t]) / 1_000_000 * SMALL_MODEL.input_price_per_m
    for t in turns
]
axes[1].plot(turns, cost_naive,  marker="o", label="Naive (new tokens only)")
axes[1].plot(turns, cost_actual, marker="s", label="Actual (full context re-sent)")
axes[1].set_title("Cumulative input cost — naive vs actual")
axes[1].set_xlabel("Agent turn")
axes[1].set_ylabel("Cumulative cost (USD)")
axes[1].legend()

plt.tight_layout()
plt.show()

multiplier = sum(input_tokens_per_turn) / (N_TURNS * TOKENS_PER_TURN)
rprint(f"Total input tokens across {N_TURNS} turns: [bold]{sum(input_tokens_per_turn)}[/bold]")
rprint(f"Cost multiplier vs naive estimate: [bold]{multiplier:.1f}×[/bold]")

<a id='section-3'></a>
## 3. Cost: Measuring Agent Spend in Practice

Understanding the theory is useful, but what you really need in production is **instrumentation** - the ability to observe actual token counts and costs on every agent run.

LangChain provides `UsageMetadataCallbackHandler` (from `langchain_core`) which intercepts LLM calls and accumulates token usage. It works with any provider that returns usage metadata, including IBM watsonx.

We will build a minimal agent with one tool and measure its token footprint across queries of varying complexity.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# A simple calculator tool — no external API needed
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result as a string.
    Only supports basic arithmetic: +, -, *, /, ** and parentheses.

    Args:
        expression: A mathematical expression, e.g. '(3 + 4) * 2'

    Returns:
        The numeric result as a string.
    """
    import ast
    try:
        tree = ast.parse(expression, mode="eval")
        allowed = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
                   ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow,
                   ast.USub, ast.UAdd)
        for node in ast.walk(tree):
            if not isinstance(node, allowed):
                return "Error: unsupported operation"
        return str(eval(compile(tree, "<string>", "eval")))
    except Exception as e:
        return f"Error: {e}"

agent = create_react_agent(llm, tools=[calculator])
rprint("[green]ReAct agent created with calculator tool[/green]")

In [ ]:
@dataclass
class RunMetrics:
    prompt: str
    input_tokens: int
    output_tokens: int
    total_tokens: int
    estimated_cost_usd: float
    latency_ms: float

def run_agent_measured(prompt: str, estimator: TokenCostEstimator) -> RunMetrics:
    """Run the agent and collect token usage + latency."""
    cb = UsageMetadataCallbackHandler()
    start = time.perf_counter()
    agent.invoke({"messages": [HumanMessage(content=prompt)]}, config={"callbacks": [cb]})
    elapsed_ms = (time.perf_counter() - start) * 1000

    input_tokens  = sum(v.get("input_tokens", 0)  for v in cb.usage_metadata.values())
    output_tokens = sum(v.get("output_tokens", 0) for v in cb.usage_metadata.values())
    total_tokens  = input_tokens + output_tokens

    cost_info = estimator.estimate(input_tokens, output_tokens)
    return RunMetrics(
        prompt=prompt,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        estimated_cost_usd=cost_info["estimated_cost_usd"],
        latency_ms=elapsed_ms,
    )

# Single run example
test_prompt = "What is (17 * 23) + (144 / 12)?"
metrics = run_agent_measured(test_prompt, SMALL_MODEL)

rprint(f"\nPrompt: [italic]{test_prompt}[/italic]")
rprint(f"  Input tokens:   [bold]{metrics.input_tokens}[/bold]")
rprint(f"  Output tokens:  [bold]{metrics.output_tokens}[/bold]")
rprint(f"  Total tokens:   [bold]{metrics.total_tokens}[/bold]")
rprint(f"  Estimated cost: [bold]${metrics.estimated_cost_usd:.6f}[/bold]")
rprint(f"  Latency:        [bold]{metrics.latency_ms:.0f} ms[/bold]")

In [ ]:
benchmark_prompts = [
    ("Simple",   "What is 5 + 3?"),
    ("Medium",   "Calculate the compound interest on $10,000 at 5% annual rate over 3 years. Use (1 + 0.05) ** 3 * 10000 - 10000."),
    ("Complex",  "I need to plan a budget. My monthly income is $5,200. Rent is $1,500, utilities are $200, groceries are $350, transport is $180, and savings should be 20% of net (income minus fixed costs). Calculate: first find fixed costs (1500+200+350+180), then net = 5200 - fixed, then savings = net * 0.20, then discretionary = net - savings."),
]

results = []
for label, prompt in benchmark_prompts:
    m = run_agent_measured(prompt, SMALL_MODEL)
    results.append({
        "complexity": label,
        "input_tokens": m.input_tokens,
        "output_tokens": m.output_tokens,
        "cost_usd": m.estimated_cost_usd,
        "latency_ms": m.latency_ms,
    })

df_bench = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = sns.color_palette("muted", 3)

axes[0].bar(df_bench["complexity"], df_bench["input_tokens"],  color=colors[0])
axes[0].bar(df_bench["complexity"], df_bench["output_tokens"], bottom=df_bench["input_tokens"], color=colors[1])
axes[0].set_title("Token usage per run")
axes[0].set_ylabel("Tokens")
axes[0].legend(["Input", "Output"], loc="upper left")

axes[1].bar(df_bench["complexity"], df_bench["cost_usd"], color=colors[2])
axes[1].set_title("Estimated cost (USD)")
axes[1].set_ylabel("USD")
for i, v in enumerate(df_bench["cost_usd"]):
    axes[1].text(i, v + v * 0.02, f"${v:.6f}", ha="center", fontsize=8)

axes[2].bar(df_bench["complexity"], df_bench["latency_ms"], color=colors[0])
axes[2].set_title("Latency (ms)")
axes[2].set_ylabel("Milliseconds")
for i, v in enumerate(df_bench["latency_ms"]):
    axes[2].text(i, v + 10, f"{v:.0f} ms", ha="center", fontsize=8)

plt.suptitle("Agent run metrics — simple vs medium vs complex", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

rprint(df_bench.to_string(index=False))

<a id='section-4'></a>
## 4. Cost: Optimisation Strategies

Once you can measure cost, you can reduce it. The five main levers are:

| Strategy | Mechanism | Typical savings |
|---|---|---|
| **Prompt compression** | Trim system prompt, summarise history | 20–50% of input tokens |
| **Model routing** | Use a small model for simple queries | 5–20× cost reduction |
| **Prompt caching** | Reuse identical prefix tokens across calls | 50–90% of repeated-context cost |
| **Output length control** | Set `max_completion_tokens`; use structured output | 30–60% output tokens |
| **Tool call batching** | Ask the model to call multiple tools in one turn | Reduces total turns |

We will implement and demonstrate the first two - the highest-impact strategies with no external dependencies.

### 4.1 Prompt compression

In a long conversation, the middle turns often contain resolved sub-tasks that no longer need to be verbatim in context. A **sliding window** keeps only the most recent N turns, replacing older content with a brief summary.

In [ ]:
def naive_token_count(text: str) -> int:
    """Rough token estimate: 1 token per ~4 characters."""
    return max(1, len(text) // 4)

def count_messages_tokens(messages: list) -> int:
    return sum(naive_token_count(m.content) for m in messages)

def compress_history(
    messages: list,
    keep_last_n: int = 4,
    summary: str = "[Earlier conversation summarised: context established, goals clarified.]"
) -> list:
    """
    Retain the system message and the most recent `keep_last_n` messages.
    Older messages are replaced with a single summary placeholder.
    """
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    non_system  = [m for m in messages if not isinstance(m, SystemMessage)]

    if len(non_system) <= keep_last_n:
        return messages

    summary_msg = AIMessage(content=summary)
    return system_msgs + [summary_msg] + non_system[-keep_last_n:]

# Build a synthetic 10-turn conversation history
synthetic_history = [
    SystemMessage(content="You are a helpful assistant specialized in financial analysis. Always show your calculations step-by-step."),
] + [
    msg for i in range(1, 6) for msg in [
        HumanMessage(content=f"Turn {i}: Calculate the ROI for a $10,000 investment with a ${i*200} annual return."),
        AIMessage(content=f"Turn {i} answer: ROI = ({i*200} / 10000) * 100 = {i*2}%. This represents a {'low' if i < 3 else 'moderate'} return."),
    ]
]

compressed = compress_history(synthetic_history, keep_last_n=4)

tokens_before = count_messages_tokens(synthetic_history)
tokens_after  = count_messages_tokens(compressed)
savings_pct   = (1 - tokens_after / tokens_before) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["Before compression", "After compression"],
              [tokens_before, tokens_after],
              color=sns.color_palette("muted", 2))
ax.bar_label(bars, labels=[f"{tokens_before} tokens", f"{tokens_after} tokens"], padding=4)
ax.set_title(f"Prompt compression: {savings_pct:.0f}% token reduction", fontsize=13)
ax.set_ylabel("Approximate token count")
plt.tight_layout()
plt.show()

rprint(f"Messages before: [bold]{len(synthetic_history)}[/bold] → after: [bold]{len(compressed)}[/bold]")
rprint(f"Token reduction: [bold]{tokens_before} → {tokens_after} ({savings_pct:.0f}% saved)[/bold]")

### 4.2 Model routing

Not every query requires a large, expensive model. A **router** classifies each incoming query by estimated complexity and dispatches it to the cheapest capable model. Simple arithmetic or fact-lookup queries go to the small model; open-ended reasoning or multi-step planning goes to the large one.

In [ ]:
def classify_complexity(query: str) -> str:
    """
    Heuristic complexity classifier.
    Returns 'simple' or 'complex'.

    In production, you would replace this with a lightweight classifier model
    or a small LLM call — the cost of that classification is far less than
    routing everything to the large model.
    """
    word_count = len(query.split())
    complex_keywords = {"compare", "analyse", "explain", "why", "how does",
                        "design", "strategy", "tradeoff", "plan", "evaluate"}
    has_complex_keyword = any(kw in query.lower() for kw in complex_keywords)
    has_multiple_steps  = query.count(".") >= 2 or query.count(";") >= 1

    if word_count > 35 or has_complex_keyword or has_multiple_steps:
        return "complex"
    return "simple"

def route_to_model(query: str) -> tuple[str, str]:
    """Returns (complexity_label, model_id)."""
    complexity = classify_complexity(query)
    if complexity == "simple":
        return "simple", "ibm/granite-4-h-small"
    return "complex", "ibm/granite-4-h-large"

# Simulate cost comparison: always-large vs routed
test_queries = [
    "What is 128 divided by 4?",
    "Convert 72 Fahrenheit to Celsius.",
    "What year was the Eiffel Tower built?",
    "Explain the tradeoffs between synchronous and asynchronous agent architectures in terms of latency, cost, and error handling.",
    "Design a cost optimisation strategy for an AI agent that processes 10,000 queries per day across three complexity tiers.",
]

rows = []
for q in test_queries:
    complexity, model_id = route_to_model(q)
    word_count = len(q.split())
    # Approximate token counts
    inp = naive_token_count(q) + 120   # query + system prompt overhead
    out = 60 + word_count // 2

    cost_routed = (SMALL_MODEL if complexity == "simple" else LARGE_MODEL).estimate(inp, out)["estimated_cost_usd"]
    cost_always_large = LARGE_MODEL.estimate(inp, out)["estimated_cost_usd"]

    rows.append({
        "query": q[:55] + ("..." if len(q) > 55 else ""),
        "routed_to": "small" if complexity == "simple" else "large",
        "cost_routed_usd": cost_routed,
        "cost_always_large_usd": cost_always_large,
        "savings_usd": cost_always_large - cost_routed,
    })

df_routing = pd.DataFrame(rows)

total_savings  = df_routing["savings_usd"].sum()
total_routed   = df_routing["cost_routed_usd"].sum()
total_unrouted = df_routing["cost_always_large_usd"].sum()
savings_pct    = (total_savings / total_unrouted) * 100

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(df_routing))
width = 0.35
ax.bar([i - width/2 for i in x], df_routing["cost_always_large_usd"], width, label="Always large model", color=sns.color_palette("muted")[3])
ax.bar([i + width/2 for i in x], df_routing["cost_routed_usd"],       width, label="Routed",            color=sns.color_palette("muted")[0])
ax.set_xticks(list(x))
ax.set_xticklabels([r["query"][:30] + "..." for r in rows], rotation=15, ha="right", fontsize=8)
ax.set_ylabel("Estimated cost (USD)")
ax.set_title(f"Model routing: {savings_pct:.0f}% cost reduction on this query mix", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

rprint(df_routing[["query", "routed_to", "cost_routed_usd", "cost_always_large_usd", "savings_usd"]].to_string(index=False))
rprint(f"\nTotal savings over {len(test_queries)} queries: [bold]${total_savings:.6f} ({savings_pct:.0f}%)[/bold]")

### 4.3 Output length control and prompt caching

**Output length control** is the simplest lever: set `max_completion_tokens` to prevent runaway generation. Pair it with structured output (JSON schemas, Pydantic models) so the model produces compact, machine-readable responses rather than lengthy prose.

**Prompt caching** is a provider-level feature where a long, stable prefix (system prompt, reference documents, tool definitions) is cached server-side. Subsequent calls that share that prefix pay only for the non-cached portion. IBM watsonx and other providers expose this via the API. The key design principle: **put the stable content first, variable content last**.

For an end-to-end observability view of cost, latency, and token usage in production, see the companion notebook: [Observability & Tracing with Langfuse](../Tracing/Tracing_Agent.ipynb).

<a id='section-5'></a>
## 5. Compute Footprint: Latency & Throughput

### Key concepts

- **Latency** is how long the user (or system) waits for a response. For LLMs it breaks into:
  - **Time-to-first-token (TTFT)**: the delay before any output begins. Dominated by prompt processing (pre-fill).
  - **Generation time**: the time to produce all output tokens. Scales linearly with output length.
- **Throughput** is how fast the model generates output, measured in **tokens per second (tok/s)**.

### Why agents are latency-sensitive

Each agent turn is a sequential LLM call. Tool calls add their own latency (network round-trip, execution). A 4-step agent task with 1.5 s per LLM call and 0.5 s tool execution time takes **≥ 8 seconds** end-to-end, even if each individual step feels fast.

```
User ──► Turn 1 (1.5 s) ──► Tool A (0.5 s) ──► Turn 2 (1.5 s) ──► Tool B (0.5 s)
      ──► Turn 3 (1.5 s) ──► Tool C (0.5 s) ──► Turn 4 (1.5 s) ──► Answer
                                                              Total: ~8 s
```

Key levers to reduce latency:
1. Smaller or distilled models (lower TTFT, faster generation)
2. Streaming (show partial output while generation continues)
3. Parallel tool calls (call multiple tools simultaneously when dependencies allow)
4. Shorter prompts (less pre-fill work)

In [ ]:
@dataclass
class LatencyResult:
    prompt_label: str
    total_ms: float
    tokens_generated: int

    @property
    def tokens_per_second(self) -> float:
        if self.total_ms == 0:
            return 0.0
        return self.tokens_generated / (self.total_ms / 1000)

def measure_latency(prompt: str, label: str) -> LatencyResult:
    """Invoke the LLM and measure wall-clock latency."""
    start = time.perf_counter()
    response = llm.invoke(prompt)
    elapsed_ms = (time.perf_counter() - start) * 1000
    tokens_out = (
        response.usage_metadata.get("output_tokens", None)
        if hasattr(response, "usage_metadata") and response.usage_metadata
        else len(response.content.split())
    )
    return LatencyResult(label, round(elapsed_ms, 1), tokens_out or 30)


latency_prompts = [
    ("Short (10 words)",   "What is the capital of France?"),
    ("Medium (30 words)",  "Explain in two sentences why prompt length affects LLM latency. Consider both the pre-fill phase and the auto-regressive generation phase."),
    ("Long (70 words)",    "I am building an AI agent pipeline for customer support. My users ask questions about order status, returns, and product specifications. The agent needs to call three separate internal APIs, aggregate results, and produce a structured JSON response within 3 seconds. Given these constraints, what approach would you recommend for minimising latency while maintaining accuracy? Please consider model size, prompt design, and tool call parallelism."),
]

latency_results = [measure_latency(prompt, label) for label, prompt in latency_prompts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels    = [r.prompt_label for r in latency_results]
latencies = [r.total_ms for r in latency_results]
tps_vals  = [r.tokens_per_second for r in latency_results]

colors = sns.color_palette("muted", 3)
axes[0].bar(labels, latencies, color=colors)
axes[0].set_title("Total latency by prompt length")
axes[0].set_ylabel("Latency (ms)")
for i, v in enumerate(latencies):
    axes[0].text(i, v + 10, f"{v:.0f} ms", ha="center", fontsize=9)

axes[1].bar(labels, tps_vals, color=colors)
axes[1].set_title("Throughput (tokens/second)")
axes[1].set_ylabel("tok/s")
for i, v in enumerate(tps_vals):
    axes[1].text(i, v + 0.5, f"{v:.1f} tok/s", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

for r in latency_results:
    rprint(f"  {r.prompt_label}: [bold]{r.total_ms:.0f} ms[/bold], {r.tokens_generated} tokens, {r.tokens_per_second:.1f} tok/s")

In [ ]:
import random
random.seed(42)

agent_steps = [
    {"label": "Plan task",         "llm_ms": 1200, "tool_ms": 0},
    {"label": "Fetch data (tool)",  "llm_ms":  900, "tool_ms": 450},
    {"label": "Analyse results",    "llm_ms": 1400, "tool_ms": 0},
    {"label": "Calculate (tool)",   "llm_ms":  800, "tool_ms": 200},
    {"label": "Generate answer",    "llm_ms": 1100, "tool_ms": 0},
]

steps      = [s["label"] for s in agent_steps]
llm_times  = [s["llm_ms"] for s in agent_steps]
tool_times = [s["tool_ms"] for s in agent_steps]
total_ms   = sum(llm_times) + sum(tool_times)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(steps))
ax.bar(x, llm_times,  label="LLM call",      color=sns.color_palette("muted")[0])
ax.bar(x, tool_times, bottom=llm_times,       label="Tool execution", color=sns.color_palette("muted")[1])
ax.set_xticks(list(x))
ax.set_xticklabels(steps, rotation=10, ha="right")
ax.set_ylabel("Time (ms)")
ax.set_title(f"Per-step latency breakdown — total: {total_ms:,} ms ({total_ms/1000:.1f} s)", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

rprint(f"Total agent task latency: [bold]{total_ms:,} ms ({total_ms/1000:.1f} s)[/bold]")
rprint(f"  LLM calls: {sum(llm_times):,} ms  |  Tool execution: {sum(tool_times):,} ms")

<a id='section-6'></a>
## 6. Compute Footprint: Memory & Context Management

### Two distinct memory concerns

1. **Process RAM**: the RAM consumed by the Python process running the agent. This covers: the LangGraph state graph, the full message history list, tool outputs, and any in-memory vector stores or caches.

2. **Context window as a memory budget**: every LLM call is bounded by a maximum context length (e.g., 128K tokens for Granite 4). Sending a large context window increases both latency and cost. Think of it as a shared resource that must be actively managed.

For API-based deployments (watsonx.ai), concern #1 is usually modest — a few hundred MB. Concern #2 is the primary operational lever.

> For self-hosted models, a third concern emerges: **GPU VRAM**. A 7B-parameter model at FP16 requires ~14 GB VRAM just for weights, before accounting for the KV cache (which grows with context length). This notebook focuses on the API-based scenario, but the context management strategies below apply equally to self-hosted deployments.

<a id='section-7'></a>
## 7. Environmental Footprint

Energy consumption and carbon emissions are increasingly important considerations for AI systems. While this is a rapidly evolving area, some grounded estimates are available:

### Energy per query

- A single LLM text query requires approximately **0.001–0.003 kWh** of energy (Google Cloud, 2025).
- For comparison: a Google web search uses ~0.0003 kWh; running a microwave for one minute uses ~0.02 kWh.
- An AI agent completing a 5-step task therefore uses **0.005–0.015 kWh** - roughly the same as sending 10–50 web search queries.

### Carbon intensity

The carbon impact depends heavily on *where* the data centre is located and what energy mix it runs on:

| Location | Typical grid carbon intensity | Notes |
|---|---|---|
| Iceland / Norway | <30 g CO₂e/kWh | Near 100% renewable hydro/geo |
| US West Coast | ~100 g CO₂e/kWh | High renewable mix |
| US national avg | ~400 g CO₂e/kWh | Mixed grid |
| Coal-heavy grids | >700 g CO₂e/kWh | Eastern Europe, parts of Asia |

Multiply energy (kWh) × carbon intensity (g CO₂e/kWh) to get per-query emissions.


### Further reading

- [MIT Technology Review: AI Energy Footprint (May 2025)](https://www.technologyreview.com/2025/05/20/1116327/ai-energy-usage-climate-footprint-big-tech/)
- [Google Cloud: Measuring the environmental impact of AI inference (2025)](https://cloud.google.com/blog/products/infrastructure/measuring-the-environmental-impact-of-ai-inference/)
- For **production monitoring** of all NFRs — cost, latency, and usage — in a single dashboard, see the companion recipe: [Observability & Tracing with Langfuse](../Tracing/Tracing_Agent.ipynb).

<a id='section-8'></a>
## 8. Summary & Reusable NFR Profiler

### Key takeaways

| Dimension | Primary metric | Biggest risk for agents | Top mitigation |
|---|---|---|---|
| **Cost** | USD per task | Context accumulation across turns | Prompt compression, model routing |
| **Latency** | ms per turn | Sequential tool calls stack up | Smaller models, parallel tool calls |
| **Throughput** | tokens/second | Blocking on long-running steps | Streaming, async execution |
| **Process memory** | MB RAM | Long message histories | Sliding window history |
| **Context budget** | % of window | Unbounded conversation history | Token-aware trimming |
| **Environmental** | kWh / CO₂e | Scale × unnecessary calls | Caching, right-sizing, routing |

### The NFR profiler

The cell below assembles all measurements from this notebook into a single reusable `NFRProfiler` class. Attach it to any LangChain LLM to collect cost, latency, and memory overhead on every run, then visualize results as a compact dashboard.

In [ ]:
@dataclass
class NFRProfiler:
    """
    Attach to any LangChain LLM to collect cost, latency, and memory per run.

    Usage:
        profiler = NFRProfiler(llm=llm, cost_estimator=SMALL_MODEL)
        profiler.run("Your prompt here")
        profiler.report()
    """
    llm: Any
    cost_estimator: TokenCostEstimator
    _results: list = field(default_factory=list)

    def run(self, prompt: str) -> dict:
        """Run the LLM and record NFR metrics."""
        mem_before = get_memory_mb()
        cb = UsageMetadataCallbackHandler()
        start = time.perf_counter()

        self.llm.invoke(prompt, config={"callbacks": [cb]})
        input_tokens  = sum(v.get("input_tokens", 0)  for v in cb.usage_metadata.values())
        output_tokens = sum(v.get("output_tokens", 0) for v in cb.usage_metadata.values())

        elapsed_ms = (time.perf_counter() - start) * 1000
        mem_after  = get_memory_mb()

        cost_info = self.cost_estimator.estimate(input_tokens, output_tokens)

        record = {
            "prompt_preview":    prompt[:50] + ("..." if len(prompt) > 50 else ""),
            "input_tokens":      input_tokens,
            "output_tokens":     output_tokens,
            "total_tokens":      input_tokens + output_tokens,
            "cost_usd":          cost_info["estimated_cost_usd"],
            "latency_ms":        round(elapsed_ms, 1),
            "memory_delta_mb":   round(mem_after - mem_before, 2),
            "tokens_per_second": round(output_tokens / max(elapsed_ms / 1000, 0.001), 1),
        }
        self._results.append(record)
        return record

    def summary_df(self) -> pd.DataFrame:
        return pd.DataFrame(self._results)

    def report(self) -> None:
        """Print a summary table and render a multi-metric dashboard."""
        df = self.summary_df()
        if df.empty:
            rprint("No runs recorded yet.")
            return

        rprint("\n[bold]NFR Profiler — Run Summary[/bold]")
        rprint(df[["prompt_preview", "total_tokens", "cost_usd",
                   "latency_ms", "tokens_per_second", "memory_delta_mb"]].to_string(index=False))

        agg = df.agg({"total_tokens": "mean", "cost_usd": "sum",
                      "latency_ms": "mean", "tokens_per_second": "mean"})
        rprint(f"\n  Avg tokens/run:  {agg['total_tokens']:.0f}")
        rprint(f"  Total cost:      ${agg['cost_usd']:.6f}")
        rprint(f"  Avg latency:     {agg['latency_ms']:.0f} ms")
        rprint(f"  Avg throughput:  {agg['tokens_per_second']:.1f} tok/s")

        fig, axes = plt.subplots(2, 2, figsize=(12, 7))
        labels = [f"Run {i+1}" for i in range(len(df))]

        axes[0, 0].bar(labels, df["total_tokens"],      color=sns.color_palette("muted")[0])
        axes[0, 0].set_title("Total tokens per run");   axes[0, 0].set_ylabel("Tokens")

        axes[0, 1].bar(labels, df["cost_usd"],          color=sns.color_palette("muted")[1])
        axes[0, 1].set_title("Estimated cost (USD)");   axes[0, 1].set_ylabel("USD")

        axes[1, 0].bar(labels, df["latency_ms"],        color=sns.color_palette("muted")[2])
        axes[1, 0].set_title("Latency per run (ms)");   axes[1, 0].set_ylabel("ms")

        axes[1, 1].bar(labels, df["tokens_per_second"], color=sns.color_palette("muted")[3])
        axes[1, 1].set_title("Throughput (tok/s)");     axes[1, 1].set_ylabel("tok/s")

        plt.suptitle("NFR Profiler Dashboard", fontsize=14, y=1.01)
        plt.tight_layout()
        plt.show()

print("NFRProfiler loaded.")

In [ ]:
profiler = NFRProfiler(
    llm=llm,
    cost_estimator=SMALL_MODEL,
)

demo_prompts = [
    "What is 9 multiplied by 7?",
    "Summarise the main tradeoffs between synchronous and asynchronous agent execution.",
    "Explain the concept of token economics for AI agents and describe three strategies to reduce cost in a production agent pipeline.",
    "You are a senior AI engineer. A business stakeholder asks why their AI agent is costing 10× more than expected after going to production. Walk through the most likely causes — context accumulation, model tier selection, tool call overhead — and provide concrete recommendations for each.",
]

for prompt in demo_prompts:
    record = profiler.run(prompt)
    rprint(f"  [green]✓[/green] {record['prompt_preview']} — {record['total_tokens']} tokens, ${record['cost_usd']:.6f}, {record['latency_ms']:.0f} ms")

profiler.report()

## Conclusion

Non-functional requirements are not optional extras for AI agents. They determine whether a system is viable in production.

The key insight from this notebook is that **agents amplify NFR risks** relative to single-turn LLM calls:

- Token costs grow super-linearly due to context re-injection on each turn.
- Latency compounds across sequential steps and tool calls.
- Memory and context budgets require active management to avoid hitting limits.

The good news is that all of these are **measurable and manageable** with the tools shown here:

| Tool / Pattern | What it addresses |
|---|---|
| `TokenCostEstimator` | Model cost awareness before deployment |
| `get_openai_callback` | Real-time token and cost instrumentation |
| `compress_history` | Input token reduction via history pruning |
| `route_to_model` | Cost reduction via dynamic model selection |
| `measure_latency` | Per-step performance benchmarking |
| `NFRProfiler` | Unified cost + latency + memory dashboard |

For production deployments, complement these patterns with an observability layer - see the [Observability & Tracing notebook](../Tracing/Tracing_Agent.ipynb) for end-to-end monitoring with Langfuse.

### References

- [IBM Granite Models](https://www.ibm.com/granite)
- [Token Economics for LLM Agents (arXiv, May 2025)](https://arxiv.org/html/2605.09104v1)
- [FinOps for AI Overview](https://www.finops.org/wg/finops-for-ai-overview/)
- [AI Cost Observability — Truefoundry (2025)](https://www.truefoundry.com/blog/ai-cost-observability)
- [Google Cloud: Measuring AI inference environmental impact (2025)](https://cloud.google.com/blog/products/infrastructure/measuring-the-environmental-impact-of-ai-inference/)
- [MIT Technology Review: AI Energy Footprint (May 2025)](https://www.technologyreview.com/2025/05/20/1116327/ai-energy-usage-climate-footprint-big-tech/)
- [LangChain Token Counting Callback (2025)](https://changelog.langchain.com/announcements/universal-token-counting-callback-for-langchain-python)